<a href="https://colab.research.google.com/github/Rohaanrz05/flyrank-ml-internship-starter-/blob/main/week%2007/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This notebook operationalizes model outputs into a human-reviewed, decision-support action playbook with explicit boundaries and reproducible exports.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_samples = 250

df_queue = pd.DataFrame({
    'url_id': [f'url_{i:04d}' for i in range(n_samples)],
    'content_archetype': np.random.choice(['Core Guide', 'Blog Post', 'Product Landing', 'FAQ / Gloss'], size=n_samples, p=[0.25, 0.45, 0.15, 0.15]),
    'staleness_days': np.random.randint(20, 450, size=n_samples),
    'impressions_30d': np.random.randint(10, 15000, size=n_samples),
    'ctr': np.random.uniform(0.005, 0.08, size=n_samples),
    'position': np.random.uniform(1.5, 45.0, size=n_samples),
    'model_risk_score': np.random.uniform(0.1, 0.95, size=n_samples)
})

def assign_action(row):
    if row['model_risk_score'] >= 0.70 and row['staleness_days'] > 180:
        return 'COMPREHENSIVE_REFRESH', 'REC_STALE_DECAY', 'High measured risk score combined with content staleness >180d'
    elif row['position'] <= 15 and row['ctr'] < 0.02 and row['model_risk_score'] >= 0.50:
        return 'TITLE_CTR_OPTIMIZE', 'REC_CTR_UNDERPERFORM', 'Top 15 ranking position with below-average CTR conversion'
    elif row['impressions_30d'] < 100 and row['staleness_days'] > 250:
        return 'CONSOLIDATE_OR_PRUNE', 'REC_LOW_IMPRESSION_TAIL', 'Persistent low impression volume on aging content asset'
    elif row['model_risk_score'] >= 0.40:
        return 'MONITOR_TREND', 'REC_MODERATE_RISK', 'Moderate risk indicator; queue for secondary inspection'
    else:
        return 'NO_ACTION', 'REC_STABLE', 'Performance metrics remain within expected variance bounds'

actions, codes, reasons = zip(*df_queue.apply(assign_action, axis=1))
df_queue['recommended_action'] = actions
df_queue['reason_code'] = codes
df_queue['human_justification'] = reasons

df_queue = df_queue.sort_values(by=['model_risk_score', 'impressions_30d'], ascending=[False, False]).reset_index(drop=True)
display(df_queue.head(10))

,url_id,content_archetype,staleness_days,impressions_30d,ctr,position,model_risk_score,recommended_action,reason_code,human_justification
0,url_0076,Product Landing,79,11921,0.029757,27.054232,0.946885,MONITOR_TREND,REC_MODERATE_RISK,Moderate risk indicator; queue for secondary i...
1,url_0103,Blog Post,365,4278,0.031561,40.104531,0.942754,COMPREHENSIVE_REFRESH,REC_STALE_DECAY,High measured risk score combined with content...
2,url_0016,Blog Post,133,4370,0.040716,35.868682,0.941790,MONITOR_TREND,REC_MODERATE_RISK,Moderate risk indicator; queue for secondary i...
3,url_0163,Blog Post,269,7124,0.062583,7.590806,0.941626,COMPREHENSIVE_REFRESH,REC_STALE_DECAY,High measured risk score combined with content...
4,url_0054,Blog Post,307,6741,0.045295,15.049083,0.940807,COMPREHENSIVE_REFRESH,REC_STALE_DECAY,High measured risk score combined with content...
5,url_0230,Product Landing,246,13538,0.063393,30.641410,0.936699,COMPREHENSIVE_REFRESH,REC_STALE_DECAY,High measured risk score combined with content...
6,url_0050,FAQ / Gloss,162,4340,0.031056,12.297850,0.934981,MONITOR_TREND,REC_MODERATE_RISK,Moderate risk indicator; queue for secondary i...
7,url_0235,Product Landing,171,13782,0.061598,22.084640,0.928495,MONITOR_TREND,REC_MODERATE_RISK,Moderate risk indicator; queue for secondary i...
8,url_0216,Blog Post,333,13174,0.022177,31.539370,0.922385,COMPREHENSIVE_REFRESH,REC_STALE_DECAY,High measured risk score combined with content...
9,url_0193,Blog Post,62,13226,0.063614,8.547604,0.919842,MONITOR_TREND,REC_MODERATE_RISK,Moderate risk indicator; queue for secondary i...


## 2. Intended use and limits

* **Intended Use:** Serves as an editorial decision-support queue to prioritize high-leverage content updates based on measured signals.
* **Operational Limits:** Predictions represent directional correlations rather than deterministic traffic guarantees; external search algorithm turbulence and brand context remain unobserved.

## 3. Human review + the no-go list

* **Mandatory Review:** Editorial evaluation of query intent relevance, domain subject accuracy, and indexing status prior to taking action.
* **Strict No-Go Automation List:**
  - No automated document deletion or URL pruning.
  - No programmatic title tag or metadata rewriting without human sign-off.
  - No bulk algorithmic 301 redirects.

## 4. Monitoring / retrain triggers

* **Feature Drift Trigger:** Population Stability Index (PSI > 0.20) on 30-day CTR or impression logs.
* **Retrain Cadence:** Quarterly retraining using client-isolated grouped splits.
* **Metric Degradation:** Performance drop exceeding 0.05 NDCG@10 / ROC-AUC on newly labeled human review batches.

## 5. Exports for the paper

*Exporting the queue to `work/outputs/` and paper figures to `work/figures/`.*

In [2]:
os.makedirs('../../work/outputs', exist_ok=True)
os.makedirs('../../work/figures', exist_ok=True)

# 1. Export Actionable Queue
queue_path = '../../work/outputs/actionable_queue.csv'
df_queue.to_csv(queue_path, index=False)
print(f'✅ Exported queue to {queue_path}')

# 2. Export Figure
fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
df_queue['recommended_action'].value_counts().plot(kind='barh', ax=ax, color='#1f77b4')
ax.set_title('Playbook Recommended Actions Distribution')
ax.set_xlabel('URL Count')
plt.tight_layout()
fig_path = '../../work/figures/action_distribution.png'
plt.savefig(fig_path)
plt.close()
print(f'✅ Exported figure to {fig_path}')

# 3. Export Metric Receipts
receipts = {
    'total_urls': len(df_queue),
    'actionable_count': int((df_queue['recommended_action'] != 'NO_ACTION').sum()),
    'top_action': str(df_queue['recommended_action'].value_counts().index[0])
}
receipt_path = '../../work/outputs/playbook_metrics.json'
with open(receipt_path, 'w') as f:
    json.dump(receipts, f, indent=2)
print(f'✅ Exported metric receipts to {receipt_path}')

✅ Exported queue to ../../work/outputs/actionable_queue.csv
✅ Exported figure to ../../work/figures/action_distribution.png
✅ Exported metric receipts to ../../work/outputs/playbook_metrics.json


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.